### 1. Import Libraries

Load the libraries required to build the NVD-based threat knowledge graph.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

### 2. Load Version-Agnostic CPE Data

Load the preprocessed CPE file in which version information has been merged. This data will be used to simplify CPE entities during NVD triple generation.

In [4]:
current_dir = Path.cwd().resolve()
BASE_DIR = next(path for path in [current_dir, *current_dir.parents] if (path / 'data').exists())

file_path = BASE_DIR / 'data' / 'processed' / 'cpe' / 'cpe_ignore_version.csv'

print(f'[INFO] Processing {file_path.name} ...')

df_cpe = pd.read_csv(
    file_path,
    usecols=[
        'part', 'vendor', 'product', 'target_sw', 'target_hw',
        'versionless_cpe', 'size'
    ]
).astype(object)
df_cpe = df_cpe.fillna('*')

print(f'[DONE] CPE data loaded successfully ({len(df_cpe)} rows)')

[INFO] Processing cpe_ignore_version.csv ...
[DONE] CPE data loaded successfully (151307 rows)


## 3. Load CVE Data

Load the preprocessed yearly CVE files from 2016 to 2025 and combine them into a single DataFrame. This merged dataset will later be split into training, validation, and test sets.

In [16]:
base_cve_dir = BASE_DIR / 'data' / 'processed' / 'cve'

df_list = []

for year in range(2016, 2025):
    file_path = base_cve_dir / f'cve-{year}.csv'

    if file_path.exists():
        print(f'[INFO] Processing {file_path.name} ...')
        df_temp = pd.read_csv(file_path).astype(object)
        df_temp = df_temp.fillna('*')
        df_list.append(df_temp)
    else:
        print(f'[WARNING] File not found: {file_path}')

if df_list:
    df_cve = pd.concat(df_list, ignore_index=True)
    print(f'[DONE] CVE data loaded successfully ({len(df_cve)} rows)')
else:
    print('[ERROR] No CVE data was loaded. Please check the file paths.')

[INFO] Processing cve-2016.csv ...
[INFO] Processing cve-2017.csv ...
[INFO] Processing cve-2018.csv ...
[INFO] Processing cve-2019.csv ...
[INFO] Processing cve-2020.csv ...
[INFO] Processing cve-2021.csv ...
[INFO] Processing cve-2022.csv ...
[INFO] Processing cve-2023.csv ...
[INFO] Processing cve-2024.csv ...
[DONE] CVE data loaded successfully (204514 rows)


### 4. Load CWE Data

Load the preprocessed CWE data for weaknesses, categories, and views. These datasets will be used to construct CWE-related entities and relations in the NVD triples.

In [5]:
base_cwe_dir = BASE_DIR / 'data' / 'processed' / 'cwe'

df_cwe = pd.read_csv(base_cwe_dir / 'cwe.csv').astype(object)
df_cwe = df_cwe.fillna('*')

df_cwe_category = pd.read_csv(base_cwe_dir / 'cwe_category.csv').astype(object)
df_cwe_category = df_cwe_category.fillna('*')

df_cwe_view = pd.read_csv(base_cwe_dir / 'cwe_view.csv').astype(object)
df_cwe_view = df_cwe_view.fillna('*')

print(f'[DONE] CWE data loaded successfully (Weakness: {len(df_cwe)}, Category: {len(df_cwe_category)}, View: {len(df_cwe_view)})')

[DONE] CWE data loaded successfully (Weakness: 944, Category: 385, View: 54)


### 5. Generate CWE Triples

Extract related weaknesses, languages, technologies, likelihood of exploit, and consequences from the CWE weakness data and convert them into triples. These triples will later be used as part of the NVD-based threat knowledge graph.

In [6]:
cwe_candidate_list = []
kg_cwe = []

for _, row in df_cwe.iterrows():
    cwe_id = row['ID']
    cwe_candidate_list.append(cwe_id)

    if row['Related_Weakness'] != '*':
        related_list = row['Related_Weakness'].split(';')
        for related in related_list:
            relation, other_id = related.split(':')
            kg_cwe.append([cwe_id, relation, f'CWE-{other_id}'])

    if row['Language'] != '*':
        language_list = row['Language'].split(';')
        for language in language_list:
            if language not in ['Language-Independent', 'Not Language-Specific']:
                kg_cwe.append([cwe_id, 'Language', language])

    if row['Technology'] != '*':
        technology_list = row['Technology'].split(';')
        for technology in technology_list:
            if technology not in ['Technology-Independent', 'Not Technology-Specific']:
                kg_cwe.append([cwe_id, 'Technology', technology])

    if row['Likelihood_Of_Exploit'] != '*':
        kg_cwe.append([cwe_id, 'LikelihoodOfExploit', row['Likelihood_Of_Exploit']])

    if row['Consequence'] != '*':
        consequence_list = row['Consequence'].split(';')
        for consequence in consequence_list:
            if consequence != 'Other':
                kg_cwe.append([cwe_id, 'Consequence', consequence])

print(f'[DONE] CWE triples generated successfully ({len(kg_cwe)} triples)')

[DONE] CWE triples generated successfully (4130 triples)


### 6. Generate CWE Category and View Triples

Extract membership relations from the CWE category and view data and convert them into triples. These triples are used to represent the structural information of CWE in the knowledge graph.

In [13]:
kg_category = []
kg_view = []

for _, row in df_cwe_category.iterrows():
    category_id = row['ID']

    if row['Has_Member'] != '*':
        member_list = row['Has_Member'].split(';')
        for member in member_list:
            kg_category.append([category_id, 'Has_Member', member])

for _, row in df_cwe_view.iterrows():
    view_id = row['ID']

    if row['Has_Member'] != '*':
        member_list = row['Has_Member'].split(';')
        for member in member_list:
            kg_view.append([view_id, 'Has_Member', member])

print(f'[DONE] CWE category triples generated successfully ({len(kg_category)} triples)')
print(f'[DONE] CWE view triples generated successfully ({len(kg_view)} triples)')

[DONE] CWE category triples generated successfully (4240 triples)
[DONE] CWE view triples generated successfully (762 triples)


### 7. Build CPE Candidate List

Extract version-agnostic CPE representations from the preprocessed CPE data and build a candidate list. This list will be used later when constructing relations between CVEs and CPEs.

In [14]:
cpe_candidate_list = df_cpe['versionless_cpe'].tolist()

print(f'[DONE] CPE candidate list created successfully ({len(cpe_candidate_list)} items)')

[DONE] CPE candidate list created successfully (151307 items)


### 8. Generate NVD Triples

Extract CPE and CWE information from the CVE data and generate internal NVD triples. In this step, CPE-CVE and CVE-CWE relations are constructed, and the lists of actually connected CVEs and CPEs are also collected.

In [17]:
cpe_candidate_set = set(cpe_candidate_list)
cwe_candidate_set = set(cwe_candidate_list)
connected_cpe_set = set()

cve_candidate_list = []
connected_cve_list = []

kg_cpe2cve = []
kg_cve2cwe = []

for _, row in df_cve.iterrows():
    cve_id = row['ID']
    cve_candidate_list.append(cve_id)
    is_connected = False

    if row['MatchingCPE'] != '*':
        related_cpes = str(row['MatchingCPE']).split(';')
        processed_cpe_set = set()

        for cpe in related_cpes:
            cpe = cpe.replace(r'\,', '.').replace(r'\:', ';').replace('"', "'")
            cpe_parts = cpe.split(':')

            if len(cpe_parts) >= 13:
                versionless_cpe = ':'.join(['cpe'] + cpe_parts[2:5] + cpe_parts[10:12])

                if versionless_cpe not in processed_cpe_set and versionless_cpe in cpe_candidate_set:
                    kg_cpe2cve.append([versionless_cpe, 'MatchingCVE', cve_id])
                    connected_cpe_set.add(versionless_cpe)
                    is_connected = True
                
                processed_cpe_set.add(versionless_cpe)

    if row['MatchingCWE'] != '*':
        related_cwes = str(row['MatchingCWE']).split(';')

        for cwe_id in related_cwes:
            if cwe_id not in ['NVD-CWE-Other', 'NVD-CWE-noinfo'] and cwe_id in cwe_candidate_set:
                kg_cve2cwe.append([cve_id, 'MatchingCWE', cwe_id])
                is_connected = True

    if is_connected:
        connected_cve_list.append(cve_id)

connected_cpe_list = list(connected_cpe_set)
kg_cve = kg_cpe2cve + kg_cve2cwe

print(f'[DONE] NVD triples generated successfully (CPE-CVE: {len(kg_cpe2cve)}, CVE-CWE: {len(kg_cve2cwe)})')

[DONE] NVD triples generated successfully (CPE-CVE: 1136638, CVE-CWE: 183698)


### 9. Save the CWE Candidate List

Save the CWE candidate list generated during preprocessing. This list is used as the full set of valid CWE candidates for later CVE→CWE evaluation.

In [18]:
output_dir = BASE_DIR / 'data' / 'processed' / 'kg' / '2024_12'
output_dir.mkdir(parents=True, exist_ok=True)

cwe_candidate_file = output_dir / 'cwe_candidate_list.txt'

cwe_candidate_list = sorted(cwe_candidate_set)

with open(cwe_candidate_file, 'w', encoding='utf-8') as f:
    for item in cwe_candidate_list:
        f.write(f'{item}\n')

print(f'[DONE] Saved: {cwe_candidate_file} ({len(cwe_candidate_list):,} items)')

[DONE] Saved: C:\Workspace\icsa-threat-kg\data\processed\kg\2024_12\cwe_candidate_list.txt (944 items)


### 10. Save Connected CPE List

Save the list of CPEs that are actually connected to CVEs during NVD triple generation. This list can be used for later analysis and follow-up preprocessing.

In [9]:
output_dir = BASE_DIR / 'data' / 'processed' / 'kg' / '2024_12'
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / 'connected_cpe_list.txt'

with open(output_file, 'w', encoding='utf-8') as f:
    for item in connected_cpe_list:
        f.write(f'{item}\n')

print(f'[DONE] Saved: {output_file} ({len(connected_cpe_list)} items)')

[DONE] Saved: C:\Workspace\icsa-threat-kg\data\processed\kg\2024_12\connected_cpe_list.txt (121760 items)


### 11. Save Connected CVE List

Save the list of CVEs that are actually connected to CWEs or CPEs during NVD triple generation. This list can be used for later analysis and follow-up preprocessing.

In [10]:
output_dir = BASE_DIR / 'data' / 'processed' / 'kg' / '2024_12'
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / 'connected_cve_list.txt'

with open(output_file, 'w', encoding='utf-8') as f:
    for item in connected_cve_list:
        f.write(f'{item}\n')

print(f'[DONE] Saved: {output_file} ({len(connected_cve_list)} items)')

[DONE] Saved: C:\Workspace\icsa-threat-kg\data\processed\kg\2024_12\connected_cve_list.txt (190824 items)


### 12. Save NVD Triples

Save the generated CPE-CVE and CVE-CWE triples as CSV files. These files will be used later for knowledge graph training and analysis.

In [11]:
output_dir = BASE_DIR / 'data' / 'processed' / 'kg' / '2024_12'
output_dir.mkdir(parents=True, exist_ok=True)

cpe2cve_file = output_dir / 'nvd_cpe2cve.csv'
cve2cwe_file = output_dir / 'nvd_cve2cwe.csv'

cpe2cve_df = pd.DataFrame(kg_cpe2cve, columns=['subject', 'predicate', 'object'])
cpe2cve_df.to_csv(cpe2cve_file, index=False, encoding='utf-8-sig')

cve2cwe_df = pd.DataFrame(kg_cve2cwe, columns=['subject', 'predicate', 'object'])
cve2cwe_df.to_csv(cve2cwe_file, index=False, encoding='utf-8-sig')

print(f'[DONE] Saved: {cpe2cve_file} ({len(cpe2cve_df)} rows)')
print(f'[DONE] Saved: {cve2cwe_file} ({len(cve2cwe_df)} rows)')

[DONE] Saved: C:\Workspace\icsa-threat-kg\data\processed\kg\2024_12\nvd_cpe2cve.csv (1136638 rows)
[DONE] Saved: C:\Workspace\icsa-threat-kg\data\processed\kg\2024_12\nvd_cve2cwe.csv (183698 rows)


### 13. Generate CPE Attribute Triples

Select only the CPEs that are actually connected to CVEs, and convert their key attributes into triples. This step is used to incorporate structural information of CPE entities into the knowledge graph.

In [12]:
connected_cpe_set = set(connected_cpe_list)

df_cpe_connected = df_cpe[df_cpe['versionless_cpe'].isin(connected_cpe_set)].copy()

kg_cpe = []

for _, row in df_cpe_connected.iterrows():
    cpe_id = row['versionless_cpe']

    kg_cpe.append([cpe_id, 'Part', row['part']])
    kg_cpe.append([cpe_id, 'Vendor', row['vendor']])
    kg_cpe.append([cpe_id, 'Product', row['product']])

    if row['target_sw'] != '*':
        kg_cpe.append([cpe_id, 'Target_sw', row['target_sw']])

    if row['target_hw'] != '*':
        kg_cpe.append([cpe_id, 'Target_hw', row['target_hw']])

print(f'[DONE] CPE attribute triples generated successfully ({len(kg_cpe)} triples)')

[DONE] CPE attribute triples generated successfully (376619 triples)


### 14. Merge Final Knowledge Graph and Check Statistics

Merge the generated CPE, CVE, CWE, category, and view triples into a single knowledge graph, and check the total numbers of triples and entities. This step serves as a basic validation to confirm that the graph has been constructed properly.

In [13]:
triples = kg_cpe + kg_cve + kg_cwe + kg_category + kg_view

print('[INFO] Running sanity check for the final knowledge graph ...')
print('--- [ Knowledge Graph Statistics ] ---')
print(f'Total triples: {len(triples):,}')

entities = set()
for head, relation, tail in triples:
    entities.add(head)
    entities.add(tail)

print(f'Total entities: {len(entities):,}')
print('--------------------------------------')
print('[DONE] Final knowledge graph statistics generated successfully')

[INFO] Running sanity check for the final knowledge graph ...
--- [ Knowledge Graph Statistics ] ---
Total triples: 1,706,087
Total entities: 446,240
--------------------------------------
[DONE] Final knowledge graph statistics generated successfully


### 15. Save and Reload the Final Knowledge Graph

Save the merged final knowledge graph triples as a CSV file, then reload them and convert them back into a list format. This step verifies the saved result and prepares the triples for later training and evaluation.

In [14]:
output_dir = BASE_DIR / 'data' / 'processed' / 'kg' / '2024_12'
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / 'nvd_threat_kg.csv'

triples_df = pd.DataFrame(triples, columns=['subject', 'predicate', 'object'])
triples_df.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f'[DONE] Saved: {output_file} ({len(triples_df)} rows)')

[DONE] Saved: C:\Workspace\icsa-threat-kg\data\processed\kg\2024_12\nvd_threat_kg.csv (1706087 rows)
